# 01b - Feast Feature Store (Remote Mode)

![Feature Store Workflow](../../docs/diagrams/01-features-workflow.png)

**Mode**: Remote client via Feast Operator gRPC services

**Capabilities**: `get_online_features`, `get_historical_features`, `list_*`

See [README.md](./README.md) for architecture and prerequisites.

---
## Step 0: Install Dependencies

In [ ]:
%pip install -q "feast==0.61.0" pandas pyarrow

---
## Step 1: Configuration

The Feast Operator automatically mounts client configuration when you link a Feature Store connection to your workbench.

In [ ]:
import os
from pathlib import Path

# Operator-provided config path (auto-mounted when Feature Store connection is configured)
FEAST_CONFIG_PATH = "/opt/app-root/src/feast-config/salesforecasting"

# Local paths for saving data
SHARED_ROOT = Path("/opt/app-root/src/shared")
DATA_DIR = SHARED_ROOT / "data"

print(f"Config path: {FEAST_CONFIG_PATH}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
# Verify config exists and show its contents
if Path(FEAST_CONFIG_PATH).exists():
    print(f"Config found: {FEAST_CONFIG_PATH}")
    print("-" * 50)
    with open(FEAST_CONFIG_PATH) as f:
        print(f.read())
else:
    print(f"ERROR: Config not found at {FEAST_CONFIG_PATH}")
    print("")
    print("To fix this:")
    print("1. Go to OpenShift AI Dashboard")
    print("2. Open your workbench settings")
    print("3. Add a Feature Store connection")
    print("4. Restart the workbench")

---
## Step 2: Connect to Feast

Initialize the Feast SDK using the operator-provided configuration.

In [ ]:
from feast import FeatureStore

store = FeatureStore(repo_path=FEAST_CONFIG_PATH)

print(f"Connected to project: {store.project}")

---
## Step 3: List Registered Objects

Query the registry to see what features are available.

In [ ]:
entities = store.list_entities()
feature_views = store.list_feature_views()
feature_services = store.list_feature_services()

print("Registered Objects:")
print(f"  Entities: {[e.name for e in entities]}")
print(f"  Feature Views: {[fv.name for fv in feature_views]}")
print(f"  Feature Services: {[fs.name for fs in feature_services]}")

if not entities:
    print("")
    print("WARNING: No features registered.")
    print("Run 01a-local.ipynb first to register features.")

---
## Step 4: Get Online Features

Retrieve **current** feature values from Redis for real-time inference.

Use case: Serving predictions in production.

In [ ]:
# Define which features to retrieve
feature_list = [
    "sales_features:weekly_sales",
    "sales_features:lag_1",
    "sales_features:lag_2",
    "sales_features:rolling_mean_4w",
    "store_features:store_type",
    "store_features:store_size",
]

# Define entities to look up
entity_rows = [
    {"store_id": 1, "dept_id": 1},
    {"store_id": 10, "dept_id": 5},
    {"store_id": 25, "dept_id": 10},
]

print(f"Retrieving {len(feature_list)} features for {len(entity_rows)} entities...")

In [ ]:
import time

t0 = time.time()
online_features = store.get_online_features(
    features=feature_list,
    entity_rows=entity_rows
).to_dict()
elapsed = (time.time() - t0) * 1000

print(f"Retrieved in {elapsed:.1f}ms")
print("-" * 50)

for key, values in online_features.items():
    print(f"{key}: {values}")

---
## Step 5: Get Historical Features

Retrieve **point-in-time** feature values for model training.

Use case: Building training datasets with correct temporal joins.

In [ ]:
import pandas as pd
from datetime import datetime, timezone

# Entity DataFrame with timestamps (point-in-time)
entity_df = pd.DataFrame([
    {"store_id": 1, "dept_id": 1, "event_timestamp": datetime(2023, 6, 1, tzinfo=timezone.utc)},
    {"store_id": 10, "dept_id": 5, "event_timestamp": datetime(2023, 6, 15, tzinfo=timezone.utc)},
    {"store_id": 25, "dept_id": 10, "event_timestamp": datetime(2023, 7, 1, tzinfo=timezone.utc)},
    {"store_id": 45, "dept_id": 14, "event_timestamp": datetime(2023, 7, 15, tzinfo=timezone.utc)},
])

print("Entity DataFrame:")
entity_df

In [ ]:
t0 = time.time()
historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "sales_features:weekly_sales",
        "sales_features:lag_1",
        "sales_features:lag_2",
        "sales_features:rolling_mean_4w",
        "store_features:store_type",
        "store_features:store_size",
    ]
).to_df()
elapsed = time.time() - t0

print(f"Retrieved {len(historical_features)} rows in {elapsed:.2f}s")
historical_features

---
## Step 6: Save for Training Pipeline

Save retrieved features for use in training notebooks.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

if len(historical_features) > 0:
    output_path = DATA_DIR / "training_features.parquet"
    historical_features.to_parquet(output_path, index=False)
    print(f"Saved: {output_path}")
    print(f"Rows: {len(historical_features)}")
    print(f"Columns: {list(historical_features.columns)}")
else:
    print("WARNING: No features to save.")
    print("Ensure data is materialized (run 01a-local.ipynb).")

---
## Done

Features retrieved successfully. Next steps:
- **Training**: Use `02_training/` notebooks with saved features
- **Inference**: Use online features in `03_inferencing/` for real-time predictions